# RAG Step 3: Retrieving Relevant Context

Once the chunks are stored, we query ChromaDB to retrieve the most relevant context for a search query.


In [ ]:
!pip install -q chromadb sentence-transformers

import chromadb
from chromadb.utils import embedding_functions

documents = [
    "RAG stands for Retrieval-Augmented Generation. It combines a retriever with a language model to answer questions using external knowledge.",
    "In a typical RAG workflow, a document is chunked, embedded, stored in a vector database, and then retrieved based on semantic similarity.",
    "ChromaDB is a popular vector database that stores embeddings and supports similarity search using metadata and filters.",
    "Large language models can hallucinate when they do not have enough context. RAG reduces this by grounding responses in relevant retrieved passages.",
    "The process usually includes loading documents, splitting them into chunks, embedding the chunks, storing them, and then querying the database."
]

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')
client = chromadb.Client()
collection = client.get_or_create_collection(
    name='rag_demo_collection',
    embedding_function=embedding_fn
)

if collection.count() == 0:
    collection.add(
        documents=documents,
        metadatas=[{'source': f'doc_{i}'} for i in range(1, len(documents) + 1)],
        ids=[f'chunk_{i}' for i in range(1, len(documents) + 1)]
    )

query = 'How does RAG reduce hallucinations?'
results = collection.query(
    query_texts=[query],
    n_results=3,
    include=['documents', 'distances', 'metadatas']
)

print('User Query:', query)
print('--- Top Retrieved Results ---')
for doc_id, doc, distance, meta in zip(results['ids'][0], results['documents'][0], results['distances'][0], results['metadatas'][0]):
    print(f'ID: {doc_id}')
    print(f'Distance: {distance:.4f}')
    print(f'Metadata: {meta}')
    print(doc)
    print('---')


## Interpretation

The retrieval step returns the passages most similar to the question. These results become the context passed to the LLM in the final RAG step.
